In [35]:
import os, sys
from dotenv import load_dotenv
load_dotenv(".env", override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

LAB_MODEL = os.getenv("LAB_MODEL", "gemini/gemini-flash-lite-latest")
_HAS_KEY = bool(os.getenv("GEMINI_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"))
PYTHON_BIN = sys.executable  # used to launch MCP servers, only relevant when USE_MCP is True

print("LLM model:", LAB_MODEL, "| API key set:", _HAS_KEY)


LLM model: gemini/gemini-flash-lite-latest | API key set: True


In [2]:
# Readiness check: confirms the libraries import cleanly
import litellm

status = {}
for name, module in [("litellm", "litellm"), ("langchain-core", "langchain_core"),
                     ("langchain-litellm", "langchain_litellm"), ("mcp (client SDK)", "mcp"),
                     ("fastmcp (server framework)", "fastmcp")]:
    try:
        __import__(module)
        status[name] = "ok"
    except Exception as e:
        status[name] = f"import failed: {e}"
status["API key"] = "set" if _HAS_KEY else "NOT set  <-- offline fallbacks will run instead"

print("Environment readiness")
print("---------------------")
for k, v in status.items():
    print(f"  {k:>28} : {v}")


c:\Users\bharath-asus\Documents\LEARNING\Documentation\Articles\agentic_ai\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment readiness
---------------------
                       litellm : ok
                langchain-core : ok
             langchain-litellm : ok
              mcp (client SDK) : ok
    fastmcp (server framework) : ok
                       API key : set


### The MCP client helper

In [3]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def call_mcp_tool(command: str, args: list[str], tool_name: str, arguments: dict, timeout: float = 20.0):
    '''Synchronous wrapper around one MCP tool call.

    command, args: how to start the server (e.g. python, ["-m", "some_server_module"]).
    tool_name, arguments: which tool to call on that server, and with what arguments.
    Returns the tool's text result, or raises if the server reports an error or times out.
    '''
    async def _call():
        server_params = StdioServerParameters(command=command, args=args)
        with open(os.devnull, "w") as devnull:
            async with stdio_client(server_params, errlog=devnull) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    result = await session.call_tool(tool_name, arguments)
                    if result.isError:
                        raise RuntimeError(str(result.content[0].text if result.content else result))
                    return result.content[0].text if result.content else None
    return asyncio.run(asyncio.wait_for(_call(), timeout=timeout))


def _unwrap_exception_group(exc):
    """anyio's TaskGroup (used internally by both stdio_client and ClientSession) can wrap
    a failure in more than one layer of ExceptionGroup. Keep unwrapping .exceptions[0] until
    we reach something that isn't a group -- that's the actual root cause. Without this, a
    fallback message would show a generic 'ExceptionGroup' instead of the real underlying
    error, making it much harder to tell what actually went wrong."""
    while hasattr(exc, "exceptions") and exc.exceptions:
        exc = exc.exceptions[0]
    return exc

print("call_mcp_tool and _unwrap_exception_group defined.")


call_mcp_tool and _unwrap_exception_group defined.


---
# Risk Analysis Agent with Tools

In [22]:
### The debt-service-coverage calculator tool

def calculate_dscr(
    ebitda_inr: float,
    existing_debt_inr: float
) -> dict:
    """
    Calculate Debt Service Coverage Ratio (DSCR).
    DSCR = EBITDA / Annual Debt Service.
    Args:
        ebitda_inr: Annual net operating income.
        existing_debt_inr: Total annual debt service.
    """

    if existing_debt_inr <= 0:
        return {
            "error": "Annual debt service must be greater than zero."
        }

    dscr = ebitda_inr / existing_debt_inr

    return {
        "net_operating_income": ebitda_inr,
        "annual_debt_service": existing_debt_inr,
        "dscr": round(dscr, 2)
    }

def interpret_dscr(dscr: float) -> dict:
    """Interpret a DSCR value."""
    if dscr < 1.0:
        interpretation = "Insufficient income to cover debt service"
        risk = "High"
    elif dscr < 1.25:
        interpretation = "Debt service coverage is relatively weak"
        risk = "Moderate"
    else:
        interpretation = "Income provides adequate coverage of debt service"
        risk = "Lower"

    return {
        "dscr": dscr,
        "risk_level": risk,
        "interpretation": interpretation
    }


In [8]:
import sqlite3
import pandas as pd

## create a SQLite database and store the CSV data into it

bureau_df = pd.read_csv("./data/wealthpilot/mock_api/bureau_reports.csv")
bank_statement_df = pd.read_csv("./data/wealthpilot/mock_api/bank_statements.csv")

conn = sqlite3.connect("applicant_details.db")

bureau_df.to_sql(
    "bureau_table",
    conn,
    if_exists="replace",
    index=False
)

bank_statement_df.to_sql(
    "bank_statement_table",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

In [9]:
# Mock credit-bureau API

def get_credit_profile(applicant_id: str) -> dict:
    """Retrieve credit bureau information for a applicant."""

    conn = sqlite3.connect("applicant_details.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT *
        FROM bureau_table
        WHERE applicant_id = ?
        """,
        (applicant_id,)
    ).fetchone()

    conn.close()

    if row is None:
        return {"error": "Applicant details not found"}

    return dict(row)

In [10]:
# Mock bank-statement API

def get_bank_statement(applicant_id: str) -> dict:
    """Retrieve bank statement information for a applicant."""

    conn = sqlite3.connect("applicant_details.db")
    conn.row_factory = sqlite3.Row

    row = conn.execute(
        """
        SELECT *
        FROM bank_statement_table
        WHERE applicant_id = ?
        """,
        (applicant_id,)
    ).fetchone()

    conn.close()

    if row is None:
        return {"error": "Applicant details not found"}

    return dict(row)

## Reliability wrappers: retry, circuit breaker, logging

- **Retry with backoff** — fail, wait a bit longer each time, try again.
- **Circuit breaker** — 3 failures *in a row* for the same tool trips it; stop even trying.
- **Logging** — every `(tool name, args, result-or-error)` recorded for later inspection.


In [11]:
import time, functools, random

TOOL_CALL_LOG: list[dict] = []

def retry_with_backoff(max_retries: int = 3, base_delay: float = 0.5):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    delay = base_delay * (2 ** attempt) + random.uniform(0, 0.1)
                    time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator


class CircuitBreakerOpen(Exception):
    pass


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3):
        self.failure_threshold = failure_threshold
        self._consecutive_failures: dict[str, int] = {}
        self._open: set[str] = set()

    def before_call(self, tool_name: str):
        if tool_name in self._open:
            raise CircuitBreakerOpen(f"Circuit open for '{tool_name}' after {self.failure_threshold} consecutive failures.")

    def record_success(self, tool_name: str):
        self._consecutive_failures[tool_name] = 0

    def record_failure(self, tool_name: str):
        n = self._consecutive_failures.get(tool_name, 0) + 1
        self._consecutive_failures[tool_name] = n
        if n >= self.failure_threshold:
            self._open.add(tool_name)


CIRCUIT_BREAKER = CircuitBreaker(failure_threshold=3)


def make_robust_tool(fn, name: str):
    """Wraps fn with circuit breaker + retry + call logging, in that order."""
    retried = retry_with_backoff(max_retries=3)(fn)

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        CIRCUIT_BREAKER.before_call(name)
        entry = {"tool": name, "args": args, "kwargs": kwargs, "ts": time.time()}
        try:
            result = retried(*args, **kwargs)
            CIRCUIT_BREAKER.record_success(name)
            entry["result"] = result
            entry["error"] = None
        except Exception as e:
            CIRCUIT_BREAKER.record_failure(name)
            entry["result"] = None
            entry["error"] = str(e)
            TOOL_CALL_LOG.append(entry)
            raise
        TOOL_CALL_LOG.append(entry)
        return result

    return wrapper

print("Reliability wrappers defined.")


Reliability wrappers defined.


## Wire the tools into LangChain (LiteLLM underneath)


In [23]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

try:
    from langchain_litellm import ChatLiteLLM
    llm = ChatLiteLLM(model=LAB_MODEL, max_retries=5)
except Exception as e:
    print(f"ChatLiteLLM unavailable ({e}); the tool-calling loop below will use its offline fallback.")
    llm = None

robust_dscr = make_robust_tool(calculate_dscr, "calculate_dscr")
robust_interpretation = make_robust_tool(interpret_dscr, "interpret_dscr")
robust_credit_profile = make_robust_tool(get_credit_profile, "get_credit_profile")
robust_bank_statement = make_robust_tool(get_bank_statement, "get_bank_statement")

@tool
def dscr(ebitda_inr: float, existing_debt_inr: float) -> dict:
    """Calculate DSCR (Debt Service Coverage Ratio) for the given EBITDA and Existing Debt Service."""
    return robust_dscr(ebitda_inr, existing_debt_inr)

@tool
def interpretation(dscr: float) -> dict:
    """Interpret the DSCR (Debt Service Coverage Ratio) value."""
    return robust_interpretation(dscr)

@tool
def credit_profile(applicant_id: str) -> dict:
    """Get credit profile of a applicant"""
    return robust_credit_profile(applicant_id)

@tool
def bank_statement(applicant_id: str) -> dict:
    """Get bank statement of a applicant"""
    return robust_bank_statement(applicant_id)

TOOLS = [dscr, interpretation, credit_profile, bank_statement]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}
print("LangChain tools declared:", list(TOOLS_BY_NAME))


LangChain tools declared: ['dscr', 'interpretation', 'credit_profile', 'bank_statement']


## The tool-call loop

In [32]:
SYSTEM_PROMPT = (
    "You are a risk-analysis agent. Use the available tools to assess the risk profile of the loan applicant. "
    "When you have everything you need, summarize the applicant's risk profile in final plain-text summary. "
)

def _fallback_risk_analysis_agent(user_request: str) -> str:
    """Deterministic, no-API demo path: calls the same real, wrapped tools in a fixed order
    (respecting USE_MCP), so the infrastructure is provable without a live key."""
    TOOL_CALL_LOG.clear()
    w = robust_dscr(3120000, 1900000)
    c = robust_interpretation(w.get('dscr', 0))
    return (
        f"Checked the DSCR of the applicant, which is {w.get('dscr', 0)}, and its interpretation is"
        f" {c.get('interpretation')}, and the risk rating for the applicant is {c.get('risk_level')}"
    )

def run_risk_analysis_agent(user_request: str, max_iterations: int = 3) -> str:
    """Run the tool-calling loop until the model stops requesting tools or max_iterations is hit."""
    if llm is None or not _HAS_KEY:
        return _fallback_risk_analysis_agent(user_request)

    model_with_tools = llm.bind_tools(TOOLS)
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_request)]

    for _ in range(max_iterations):
        ai_msg = model_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not getattr(ai_msg, "tool_calls", None):
            return ai_msg.content

        for tc in ai_msg.tool_calls:
            tool_fn = TOOLS_BY_NAME[tc["name"]]
            try:
                result = tool_fn.invoke(tc["args"])
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return "Stopped: reached max_iterations without a final answer."

print("run_risk_analysis_agent defined.")


run_risk_analysis_agent defined.


In [33]:
TOOL_CALL_LOG.clear()
final_answer = run_risk_analysis_agent(
    "Applicant ID: ASH-A-00000. Business name is Zavala Enterprises, and they are in Light Engineering sector."
    "EBITDA in INR is 1286492, existing debt service is 1386130. And they are GST registered."
)

In [34]:
print(final_answer)

**Risk Analysis Summary for Zavala Enterprises (Applicant ID: ASH-A-00000)**

* **Business Sector:** Light Engineering
* **GST Status:** Registered

### Financial & Debt Metrics:
* **EBITDA:** ₹1,286,492
* **Existing Debt Service:** ₹1,386,130
* **Debt Service Coverage Ratio (DSCR):** 0.93 (High Risk - Insufficient earnings to cover existing debt obligations)

### Credit Profile:
* **Bureau Score:** 374 (Critically low, indicating severe historical credit distress)
* **Credit Enquiries (Last 6 Months):** 5 (High inquiry frequency, signalling liquidity/credit-seeking pressure)
* **Delinquencies / DPD:** No recent 30+ DPD in the last 12 months and no 90+ DPD ever recorded, but active trade lines stand at 0.
* **Written-Off Amount:** ₹900,000 (Significant past defaults/write-offs)

### Bank Statement Analysis (May 2025):
* **Inflows:** ₹7,140,941
* **Outflows:** ₹7,849,111 (Outflows exceed inflows for the period)
* **Closing Balance:** ₹2,868,895
* **Average Daily Balance:** ₹2,294,870
* 